In [1]:
import pandas as pd

from libs.SVDModel import SVDModel
from libs.utils import *
from libs.DataLoader import Loader

In [14]:
loader = Loader('ur0.01_ir0.01', 64)
(train_df, val_df), _, _= loader.load_data()

Load data


KeyboardInterrupt: 

In [13]:
train_df = train_df[train_df[TARGET] != 0]  # Interaction: positive/negative
val_df = val_df[val_df[TARGET] > 0]  # Interaction: positive

print(f"Train: {len(train_df):_} Val: {len(val_df):_}")

Train: 62_578 Val: 1_970


In [5]:
model = SVDModel(60).fit(train_df)

In [6]:
predict = model.batch_recommend_users_for_items(
    item_ids=val_df[ITEM].unique().tolist(),
    n_recommendations=100,
    batch_size=1000
)

Predict batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
submission = pd.DataFrame([(item, users) for item, (users, score) in predict.items()], columns=[ITEM, USER])
true_positive_reactions = val_df.groupby(ITEM).agg({USER: list}).reset_index()

In [12]:
print(f"Val DCG: {DCG(submission, true_positive_reactions):.3f}")
print(f"Val NDCG: {NDCG(submission, true_positive_reactions):.3f}")

Val DCG: 0.067
Val NDCG: 0.044
